In [68]:
import os
import numpy as np
import pandas as pd
from reComBat import reComBat
import seaborn as sns
import os
os.chdir('d:/flujo_portables/NormativeModel_EEG')
from Functions.functions import apply_pca_umap
from Functions.graphs_functions import plot_spearman_correlation


In [63]:
df = pd.read_csv(r"D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\neuroharmonize\unharmonize_rel_a1_O1_age.csv")
data = pd.concat([df.iloc[:, 8:14], df.iloc[:, 19:-4]], axis=1)

metadata  = df.iloc[:,[1,2,3,6,7,14,15]]

In [64]:
model = reComBat(parametric=True,            # use parametric or non-parametric empirical Bayes method. 
                                             # The parametric method is significantly faster, whereas the 
                                             # non-parametric method is more flexible.
                 model='ridge',              # The regression model to be used. 
                                             # In our experience pure ridge regression performs best for singular design matrices 
                                             # and pure linear regression is best for non=singular matrices.
                 config={'alpha':1e-9},      # Optional arguments for the regression model. 
                                             # We tend to use a tiny regularisation parameter. 
                                             # This has also been cnfirmed by CV.
                 conv_criterion=1e-4,        # The convergence criterion for the empirical Bayes optimisation.
                                             # This value works well in practise.
                 max_iter=1000,              # The maximum number of iterations to stop if convergence is not reached.
                                             # This may also be useful for smaller convergence criteria.
                 n_jobs=1,                   # This parameter is only useful in non-parametric optimisation.
                                             # The non-parametric optimisation is very slow, but can be parallelised easily.
                                             # Set this to the number of CPUs on your machine for significant speed ups.
                 mean_only=False,            # Adjust the mean of your data only (not the variance).
                                             # This can be useful for single sample batches (where the variance is infinite)
                 optimize_params=True,       # If False no empirical Bayes optimisation is performed.
                 reference_batch=None,       # If a reference batch is present (e.g. a batch which is considered "batch effect free")
                                             # it can be set such that all data is adjuste dwith respect to this reference batch.
                 verbose=True                # Turn log messages on or off
)

In [65]:
# Ajustamos y transformamos los datos
X_harmonized = model.fit_transform(data, metadata.SITE, X=metadata.drop('SITE', axis=1))
# Convertimos a DataFrame si es necesario
df_harmonized = pd.DataFrame(X_harmonized, columns=data.columns)

print(df_harmonized.head())

[reComBat] 2025-03-11 10:31:39,377 Starting to fot reComBat.
[reComBat] 2025-03-11 10:31:39,525 Fit the linear model.
[reComBat] 2025-03-11 10:31:42,966 Starting the empirical parametric optimisation.
[reComBat] 2025-03-11 10:31:43,025 Optimisation finished.
[reComBat] 2025-03-11 10:31:43,027 reComBat is fitted.
[reComBat] 2025-03-11 10:31:43,031 Starting to transform.
[reComBat] 2025-03-11 10:31:43,240 Transform finished.


   power_delta_O1  power_theta_O1  power_BGF1_O1  power_BGF2_O1  \
0        0.056199        0.029764       0.036130       0.357102   
1        0.077748        0.027836       0.038174       0.385636   
2        0.131333        0.063271       0.092098       0.241221   
3        0.027446        0.023137       0.024621       0.265488   
4        0.069030        0.052385       0.053105       0.360636   

   power_BGF3_O1  power_beta_O1  fooof_power_delta_O1  fooof_power_theta_O1  \
0       0.372939       0.142685              0.080676              0.030763   
1       0.329915       0.135510              0.065909              0.036126   
2       0.181250       0.285645              0.142480              0.068042   
3       0.600444       0.053683              0.032611              0.019853   
4       0.328477       0.131184              0.071568              0.046932   

   fooof_power_BGF1_O1  fooof_power_BGF2_O1  fooof_power_BGF3_O1  \
0             0.036086             0.400526           